# 🤖 Baselines — Kaggle Training (Step 4/8)
**Trains 10 baseline models sequentially with crash recovery.**

## ⚙️ Setup
1. **Datasets** — Add all 12 datasets (Input → Add Data)
2. **Secret** — Add `HF_TOKEN` secret
3. **Accelerator** — GPU T4 x2
4. **Run All**

> Note: This notebook runs a lot of models. If Kaggle session times out (12h limit), just re-run it. It will automatically resume from the last trained epoch of whichever baseline it was working on.

**Sequence**: Tiny → Base → Large → **Baselines** → Ablation → PaperEvals → LOGO → Outputs

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 0: Clone Repo + Install Dependencies
# ═══════════════════════════════════════════════════════════
import subprocess, sys, os
from pathlib import Path

REPO_URL  = "https://github.com/MIHMahmudEli/ai-image-detection-research.git"
CLONE_DIR = Path("/kaggle/working/ai-image-detection-research")

if not CLONE_DIR.exists():
    print("Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, str(CLONE_DIR)], check=True)
else:
    print("Pulling latest...")
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull", "--rebase"], check=False)

os.chdir("/kaggle/working")
sys.path.insert(0, str(CLONE_DIR / "model"))

subprocess.run([sys.executable, "-m", "pip", "install",
    "huggingface_hub", "open_clip_torch", "scipy", "scikit-learn", "timm",
    "-q", "--disable-pip-version-check"], check=False)

print(f"Project root: {CLONE_DIR}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 1: Imports & Environment
# ═══════════════════════════════════════════════════════════
import os, sys, math, json, time, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.notebook import tqdm

sys.path.insert(0, str(Path("/kaggle/working/ai-image-detection-research/model")))
from src.kaggle_utils import KaggleEnv

env = KaggleEnv(project_root_search=True)
PROJECT_ROOT = env.project_root
os.chdir(env.working_dir)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SMOKE_TEST = not torch.cuda.is_available()
print(f"Device: {device} | SMOKE: {SMOKE_TEST}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 2: Config & Dataloaders
# ═══════════════════════════════════════════════════════════
from src.dataset import create_split_dataloaders
from src.config import Config

cfg = Config()
NUM_EPOCHS  = 1   if SMOKE_TEST else 20
IMAGE_SIZE  = 224 if SMOKE_TEST else 384
BATCH_SIZE  = 8   if SMOKE_TEST else 64
NUM_WORKERS = 0   if SMOKE_TEST else 4
MAX_SAMPLES = 600 if SMOKE_TEST else None

cfg.training.epochs         = NUM_EPOCHS
cfg.training.image_size     = IMAGE_SIZE
cfg.training.batch_size     = BATCH_SIZE

_manifest = PROJECT_ROOT / "dataset" / "metadata" / "train_manifest.csv"
if not _manifest.exists() or _manifest.stat().st_size < 1000:
    env.download_manifest(_manifest)
if not _manifest.exists() or _manifest.stat().st_size < 1000:
    env.rebuild_manifest_from_kaggle(_manifest)
if _manifest.exists(): cfg.dataset.metadata_paths = [str(_manifest)]

_split_name = "split_indices_smoke.json" if SMOKE_TEST else "split_indices.json"
_split_path = PROJECT_ROOT / "dataset" / "metadata" / _split_name
if not _split_path.exists() and not SMOKE_TEST and env.hf_token:
    try:
        from huggingface_hub import hf_hub_download
        import shutil
        _dl = hf_hub_download(repo_id=env.hf_manifest_repo, filename="split_indices.json", repo_type="model", token=env.hf_token)
        shutil.copy2(_dl, _split_path)
    except Exception as e: print(f"Warning: could not download split_indices from HF: {e}")

train_loader, val_loader, test_loader = create_split_dataloaders(
    root_dir=str(PROJECT_ROOT), metadata_paths=cfg.dataset.metadata_paths,
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, size=IMAGE_SIZE,
    val_split=0.10, test_split=0.10, seed=SEED, use_weighted_sampler=True,
    split_index_path=str(_split_path), max_samples=MAX_SAMPLES
)
print(f"Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 3: Define Baselines
# ═══════════════════════════════════════════════════════════
from src.baselines import (
    SimpleCNN, LightViT, count_parameters,
    resnet18, resnet50, efficientnet_b0, vit_b_16, swin_t,
    CLIPBaseline, FreqDetect, deit_small
)

BASELINE_REGISTRY = {
    'SimpleCNN':      lambda: SimpleCNN(),
    'LightViT':       lambda: LightViT(img_size=IMAGE_SIZE, depth=4, num_heads=4, embed_dim=192),
    'ResNet-18':      lambda: resnet18(),
    'ResNet-50':      lambda: resnet50(),
    'EfficientNet-B0': lambda: efficientnet_b0(),
    'ViT-B/16':       lambda: vit_b_16(img_size=IMAGE_SIZE),
    'Swin-T':         lambda: swin_t(),
    'DeiT-S':         lambda: deit_small(img_size=IMAGE_SIZE),
    'CLIP':           lambda: CLIPBaseline(img_size=IMAGE_SIZE),
    'FreqDetect':     lambda: FreqDetect(img_size=IMAGE_SIZE),
}

MODELS_TO_TRAIN = list(BASELINE_REGISTRY.keys())
if SMOKE_TEST: MODELS_TO_TRAIN = ['SimpleCNN', 'ResNet-18', 'FreqDetect']
print(f"Will train: {MODELS_TO_TRAIN}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 4: Train All Baselines Sequentially
# ═══════════════════════════════════════════════════════════
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, LinearLR, SequentialLR
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

AMP       = torch.cuda.is_available()
scaler    = torch.amp.GradScaler("cuda", enabled=AMP)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
all_metrics = []

for model_name in MODELS_TO_TRAIN:
    safe_name = model_name.lower().replace('/', '_').replace('-', '_')
    print(f"\n{'='*70}\nTraining {model_name} (Safe name: {safe_name})\n{'='*70}")

    model = BASELINE_REGISTRY[model_name]().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.training.lr, weight_decay=cfg.training.weight_decay)
    warmup = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=min(500, len(train_loader)))
    cosine = CosineAnnealingWarmRestarts(optimizer, T_0=NUM_EPOCHS*len(train_loader), T_mult=2, eta_min=1e-6)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[min(500, len(train_loader))])

    ckpt_dir = PROJECT_ROOT / "model" / "checkpoints" / safe_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    start_epoch = 0; best_acc = 0.0; best_epoch = -1

    # Check if already fully trained (metrics.json exists)
    results_dir = PROJECT_ROOT / "paper" / "result" / "full_scale" / safe_name
    results_dir.mkdir(parents=True, exist_ok=True)
    metrics_file = results_dir / "metrics.json"
    if metrics_file.exists():
        print(f"{model_name} already completed. Skipping training.")
        with open(metrics_file) as f: all_metrics.append(json.load(f))
        continue

    # Resume logic
    resume_ckpt = env.find_resume_checkpoint(ckpt_dir)
    if not resume_ckpt: resume_ckpt = env.download_latest_checkpoint(ckpt_dir, safe_name)
    if resume_ckpt:
        state = env.load_checkpoint(resume_ckpt, model, optimizer, scheduler, scaler, device=device)
        if state:
            start_epoch, best_acc = state["epoch"] + 1, state["best_acc"]
            history = state["history"] or history
            print(f"  Resumed epoch {start_epoch}, best={best_acc:.2f}%")

    # Training loop for this baseline
    for epoch in range(start_epoch, NUM_EPOCHS):
        model.train()
        total_loss = correct = total = 0
        pbar = tqdm(train_loader, desc=f"{model_name} Ep {epoch+1}/{NUM_EPOCHS}")
        for images, labels in pbar:
            try:
                images, labels = images.to(device), labels.to(device)
                with torch.amp.autocast("cuda", enabled=AMP):
                    loss = criterion(model(images), labels)
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
                optimizer.zero_grad(); scheduler.step()
                total_loss += loss.item()
                correct += (loss > -1).sum().item()*0 # Hack: dummy update
                # Proper metrics:
                with torch.no_grad():
                    preds = model(images).argmax(-1)
                    correct += (preds == labels).sum().item(); total += labels.size(0)
                pbar.set_postfix({"loss": f"{total_loss/max(1,total//BATCH_SIZE):.4f}", "acc": f"{correct/total*100:.2f}%"})
            except Exception as e: print(f"  Skip batch: {e}"); optimizer.zero_grad()

        train_acc = correct / total * 100
        history["train_acc"].append(train_acc); history["train_loss"].append(total_loss / len(train_loader))

        model.eval(); val_loss = val_correct = val_total = 0
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"{model_name} Val", leave=False):
                try:
                    images, labels = images.to(device), labels.to(device)
                    logits = model(images)
                    val_loss += criterion(logits, labels).item()
                    val_correct += (logits.argmax(-1) == labels).sum().item(); val_total += labels.size(0)
                except Exception: pass

        val_acc = val_correct / val_total * 100
        history["val_acc"].append(val_acc); history["val_loss"].append(val_loss / len(val_loader))
        print(f"  Ep {epoch+1}: train={train_acc:.2f}% val={val_acc:.2f}%")

        if val_acc > best_acc:
            best_acc = val_acc; best_epoch = epoch + 1
            torch.save(model.state_dict(), ckpt_dir / "best.pt")
            print(f"  ⭐ New best: {best_acc:.2f}%")

        if (epoch + 1) % 5 == 0 or epoch == NUM_EPOCHS - 1:
            ckpt_path = ckpt_dir / f"checkpoint_epoch_{epoch+1}.pt"
            env.save_checkpoint(ckpt_path, model, optimizer, scheduler, scaler, epoch, best_acc, history)
            env.upload_checkpoint(ckpt_path, safe_name)

    print(f"\n{model_name} done. Best val acc: {best_acc:.2f}%")

    # -- Test Set Evaluation --
    if start_epoch < NUM_EPOCHS: # Only evaluate if we actually trained/finished
        model.load_state_dict(torch.load(ckpt_dir / "best.pt", weights_only=True))
    model.eval(); all_labels, all_probs = [], []
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc=f"{model_name} Test"):
            probs = F.softmax(model(images.to(device)), dim=-1)
            all_labels.extend(labels.cpu().numpy()); all_probs.extend(probs[:, 1].cpu().numpy())

    y_true, y_score = np.array(all_labels), np.array(all_probs)
    y_pred = (y_score >= 0.5).astype(int)

    acc = (y_pred == y_true).mean() * 100
    prec = precision_score(y_true, y_pred, zero_division=0) * 100
    rec = recall_score(y_true, y_pred, zero_division=0) * 100
    f1 = f1_score(y_true, y_pred, zero_division=0) * 100
    auc = roc_auc_score(y_true, y_score)

    metrics = {
        "model": model_name, "params": count_parameters(model),
        "best_val_acc": round(best_acc, 2), "best_epoch": best_epoch,
        "test_accuracy": round(acc, 2), "test_precision": round(prec, 2),
        "test_recall": round(rec, 2), "test_f1": round(f1, 2), "test_auc": round(auc, 4),
    }
    all_metrics.append(metrics)
    with open(results_dir / "metrics.json", "w") as f: json.dump(metrics, f, indent=2)
    with open(results_dir / "history.json", "w") as f: json.dump(history, f, indent=2)
    
    # Upload this baseline's results
    env.upload_to_hf(results_dir / "metrics.json", env.hf_results_repo, f"results/baselines/{safe_name}/metrics.json")
    env.upload_to_hf(results_dir / "history.json", env.hf_results_repo, f"results/baselines/{safe_name}/history.json")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Cell 5: Summary Table of All Baselines
# ═══════════════════════════════════════════════════════════
if all_metrics:
    df = pd.DataFrame(all_metrics).set_index("model")
    print("\n=== BASELINE COMPARISON ===")
    print(df.to_string())
    
    mode = "verify" if SMOKE_TEST else "full_scale"
    summary_path = PROJECT_ROOT / "paper" / "result" / mode / "baseline_summary.csv"
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(summary_path)
    
    env.upload_to_hf(summary_path, env.hf_results_repo, "results/baselines/baseline_summary.csv")
    print(f"\nSummary saved and uploaded. Proceed to 05_ablation.ipynb")